# 🚀 Agent Evaluation Starter

This notebook evaluates **any Azure AI Foundry agents** defined in `agents.yaml` using the **Azure AI Evaluation SDK**.

**Two-phase approach**
1. **Phase 1** collects agent responses and caches them to JSONL.
2. **Phase 2** scores the saved responses with configurable evaluators.

The notebook is intentionally **generic and config-driven** so it can be used for customer service bots, code review agents, sales assistants, and more.


## ⚙️ Section 1: Configuration

Load environment variables, resolve the repo root, and read agent + dataset settings from `agents.yaml`.


In [ ]:
import os, yaml, json, hashlib, time, asyncio
from pathlib import Path
from dotenv import load_dotenv

# Repo root resolution
REPO_ROOT = Path(os.getcwd())
if (REPO_ROOT / "notebooks").exists():
    pass
elif (REPO_ROOT.parent / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")

# Load agents.yaml
config_path = REPO_ROOT / "agents.yaml"  # ⬇️ CUSTOMIZE: point to a different config file if needed
if not config_path.exists():
    raise FileNotFoundError(
        "agents.yaml not found! Copy agents.yaml.template to agents.yaml and fill in your agent details."
    )

with open(config_path, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

AGENTS = CONFIG["agents"]
PROJECT = CONFIG["project"]
DATASET_CFG = CONFIG["dataset"]
EVALUATOR_CFG = CONFIG.get("evaluators", {})
OUTPUT_CFG = CONFIG.get("output", {})

ENDPOINT = os.environ[PROJECT["endpoint_env"]]
EVAL_ENDPOINT = os.environ.get(PROJECT.get("eval_endpoint_env", ""), ENDPOINT)
MODEL = PROJECT.get("model", "gpt-4o")  # ⬇️ CUSTOMIZE: fallback only if not set in agents.yaml

print(f"✅ Loaded {len(AGENTS)} agents from agents.yaml")
for name, info in AGENTS.items():
    print(f"   • {name} ({info['role']}): {info['description']}")


## 📦 Section 2: Imports

Import the Azure OpenAI and Azure AI Evaluation SDK clients used in both phases.


In [ ]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.evaluation import (
    evaluate,
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    F1ScoreEvaluator,
    ViolenceEvaluator,
)
import pandas as pd
from IPython.display import display

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
oai = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2025-04-01-preview",
)
print("✅ Azure OpenAI client initialized")


## 📚 Section 3: Load or Preview Dataset

Load the JSONL dataset configured in `agents.yaml` and preview a few rows before running evaluation.


In [ ]:
dataset_path = REPO_ROOT / DATASET_CFG["path"]  # ⬇️ CUSTOMIZE via agents.yaml -> dataset.path
df = None

if not dataset_path.exists():
    print(f"⚠️  Dataset not found at {dataset_path}")
    print("   Run: python _test_pack/create_dataset.py")
    print("   Or create your own JSONL with columns: query, context, ground_truth")
else:
    df = pd.read_json(dataset_path, lines=True)
    print(f"✅ Loaded {len(df)} rows from {dataset_path.name}")
    display(df.head())


## �� Section 4: Phase 1 — Collect Agent Responses

Call each configured Foundry agent, save responses to JSONL, and reuse cached results when the agent config + dataset fingerprint has not changed.


In [ ]:
results_dir = REPO_ROOT / OUTPUT_CFG.get("results_dir", "eval_results")  # ⬇️ CUSTOMIZE via agents.yaml -> output.results_dir
results_dir.mkdir(exist_ok=True)
precomputed_path = results_dir / OUTPUT_CFG.get("precomputed_file", "precomputed.jsonl")
cache_meta_path = results_dir / ".cache_meta.json"

# Build fingerprint
fingerprint_src = json.dumps(
    {k: v for k, v in sorted(AGENTS.items())}, sort_keys=True
) + ENDPOINT + (
    dataset_path.read_bytes().decode("utf-8", errors="replace") if dataset_path.exists() else ""
)
fingerprint = hashlib.sha256(fingerprint_src.encode()).hexdigest()[:16]

# Check cache
use_cache = False
if cache_meta_path.exists() and precomputed_path.exists():
    meta = json.loads(cache_meta_path.read_text(encoding="utf-8"))
    if meta.get("fingerprint") == fingerprint:
        use_cache = True
        print(f"✅ Cache valid — reusing {precomputed_path.name} ({meta.get('row_count', '?')} rows)")

if not use_cache:
    if df is None:
        raise FileNotFoundError(
            f"Dataset file not found at {dataset_path}. Create it first or update agents.yaml."
        )

    query_col = DATASET_CFG["columns"]["query"]
    context_col = DATASET_CFG["columns"].get("context", "context")
    ground_truth_col = DATASET_CFG["columns"].get("ground_truth", "ground_truth")

    if query_col not in df.columns:
        raise KeyError(f"Dataset is missing required query column: {query_col}")

    def call_agent(agent_name: str, prompt: str, retries: int = 3) -> str:
        for attempt in range(retries):
            try:
                resp = oai.responses.create(
                    model=MODEL,
                    input=prompt,
                    extra_body={
                        "agent_reference": {
                            "name": agent_name,
                            "type": "agent_reference",
                        }
                    },
                )
                return resp.output_text
            except Exception as e:
                if "429" in str(e) and attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return f"[ERROR] {e}"

    rows = df.to_dict("records")
    results = []
    total = len(rows) * len(AGENTS)
    done = 0

    for row in rows:
        query = row[query_col]
        for agent_name, agent_info in AGENTS.items():
            response = call_agent(agent_name, query)
            results.append({
                "agent": agent_name,
                "role": agent_info["role"],
                "query": query,
                "response": response,
                "context": row.get(context_col, ""),
                "ground_truth": row.get(ground_truth_col, ""),
            })
            done += 1
            if done % 5 == 0 or done == total:
                print(f"   Progress: {done}/{total}")

    with open(precomputed_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    cache_meta_path.write_text(json.dumps({
        "fingerprint": fingerprint,
        "row_count": len(results),
        "agents": list(AGENTS.keys()),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✅ Phase 1 complete — {len(results)} responses saved to {precomputed_path.name}")
elif df is None:
    print("ℹ️  Dataset preview skipped, but cached responses are available for evaluation.")


## 🧪 Section 5: Phase 2 — Run Evaluators

Build evaluators from `agents.yaml`, then score the saved JSONL responses. To reduce rate-limit pressure, the notebook runs evaluators in two batches when needed.


In [ ]:
# Build evaluator dict based on agents.yaml config
azure_ai_project = {
    "subscription_id": os.environ.get("AZURE_SUBSCRIPTION_ID", ""),
    "resource_group_name": os.environ.get("AZURE_RESOURCE_GROUP", ""),
    "project_name": os.environ.get("AZURE_PROJECT_NAME", ""),
}

model_config = {
    "azure_endpoint": EVAL_ENDPOINT,
    "azure_deployment": MODEL,
    "api_version": "2025-04-01-preview",
}

evaluator_map = {
    "groundedness": GroundednessEvaluator,
    "relevance": RelevanceEvaluator,
    "coherence": CoherenceEvaluator,
    "fluency": FluencyEvaluator,
    "similarity": SimilarityEvaluator,
    "f1_score": F1ScoreEvaluator,
    "violence": ViolenceEvaluator,
}

quality_names = {"groundedness", "relevance", "coherence", "fluency"}
quality_evaluators = {}
secondary_evaluators = {}

for name, cls in evaluator_map.items():
    if EVALUATOR_CFG.get(name, False):
        try:
            if name == "f1_score":
                evaluator = cls()
            elif name == "violence":
                evaluator = cls(
                    azure_ai_project=azure_ai_project,
                    credential=credential,
                )
            else:
                evaluator = cls(model_config=model_config, credential=credential)

            if name in quality_names:
                quality_evaluators[name] = evaluator
            else:
                secondary_evaluators[name] = evaluator
        except Exception as e:
            print(f"⚠️  Could not init {name}: {e}")

active_evaluators = {**quality_evaluators, **secondary_evaluators}
print(f"✅ {len(active_evaluators)} evaluators active: {list(active_evaluators.keys())}")

if not active_evaluators:
    raise ValueError("No evaluators are enabled. Update the evaluators block in agents.yaml.")

def evaluator_mapping(name: str):
    if name == "groundedness":
        return {
            "query": "${data.query}",
            "response": "${data.response}",
            "context": "${data.context}",
        }
    if name in ("similarity", "f1_score"):
        return {
            "response": "${data.response}",
            "ground_truth": "${data.ground_truth}",
        }
    return {
        "query": "${data.query}",
        "response": "${data.response}",
        "context": "${data.context}",
        "ground_truth": "${data.ground_truth}",
    }

def normalize_rows(result_obj):
    return result_obj.get("rows", []) if isinstance(result_obj, dict) else getattr(result_obj, "rows", [])

def normalize_metrics(result_obj):
    return result_obj.get("metrics", {}) if isinstance(result_obj, dict) else getattr(result_obj, "metrics", {})

batch_results = []

if quality_evaluators:
    print(f"🚀 Running batch 1 with {list(quality_evaluators.keys())}")
    batch_results.append(
        evaluate(
            data=str(precomputed_path),
            evaluators=quality_evaluators,
            evaluator_config={
                name: evaluator_mapping(name)
                for name in quality_evaluators
            },
        )
    )

if secondary_evaluators:
    if quality_evaluators:
        print("⏳ Waiting 30s before batch 2 to reduce rate-limit pressure...")
        time.sleep(30)
    print(f"🚀 Running batch 2 with {list(secondary_evaluators.keys())}")
    batch_results.append(
        evaluate(
            data=str(precomputed_path),
            evaluators=secondary_evaluators,
            evaluator_config={
                name: evaluator_mapping(name)
                for name in secondary_evaluators
            },
        )
    )

merged_rows = []
if batch_results:
    merged_rows = [dict(row) for row in normalize_rows(batch_results[0])]
    for extra_result in batch_results[1:]:
        extra_rows = normalize_rows(extra_result)
        for idx, row in enumerate(extra_rows):
            if idx < len(merged_rows):
                merged_rows[idx].update(row)
            else:
                merged_rows.append(dict(row))

eval_result = {
    "rows": merged_rows,
    "metrics": {
        key: value
        for result_obj in batch_results
        for key, value in normalize_metrics(result_obj).items()
    },
}

print("✅ Phase 2 complete — evaluation scores computed")


## 📈 Section 6: Parse Results

Convert the SDK output to a DataFrame and detect the score columns for downstream reporting.


In [ ]:
eval_rows = eval_result.get("rows", [])
eval_df = pd.DataFrame(eval_rows)
print(f"✅ {len(eval_df)} scored rows")

def is_score_col(col_name: str) -> bool:
    lower = col_name.lower()
    excluded_suffixes = (
        "_reason",
        "_result",
        "_threshold",
        "_model",
        "_prompt_tokens",
        "_completion_tokens",
        "_total_tokens",
    )
    return (
        not lower.endswith(excluded_suffixes)
        and (
            any(lower.startswith(name) for name in active_evaluators)
            or lower.endswith("score")
        )
    )

# Extract score columns
score_cols = [c for c in eval_df.columns if is_score_col(c)]
print(f"   Score columns: {score_cols}")

if not eval_df.empty:
    display(eval_df.head())


## 📊 Section 7: Dashboard

Plot average evaluator scores by agent and show a compact summary table.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if score_cols and "agent" in eval_df.columns:
    agent_scores = eval_df.groupby("agent")[score_cols].mean(numeric_only=True)

    fig, axes = plt.subplots(1, min(len(score_cols), 4), figsize=(5 * min(len(score_cols), 4), 5))
    if len(score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, score_cols[:4]):
        agent_scores[col].plot(
            kind="bar",
            ax=ax,
            color=["#0078D4", "#50E6FF", "#00B294", "#FFB900"][:len(AGENTS)],
        )
        ax.set_title(col.replace(".", " ").title())
        ax.set_ylabel("Score")
        ax.set_ylim(0, 5.5)
        ax.tick_params(axis="x", rotation=45)

    plt.suptitle("Agent Evaluation Dashboard", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Summary table
    print("\n📊 Average Scores by Agent:")
    display(agent_scores.round(2))
else:
    print("⚠️  No score columns found — check evaluator output")


## 💾 Section 8: Export Results

Save the scored rows as JSON and CSV for reuse outside the notebook.


In [ ]:
eval_results_path = results_dir / OUTPUT_CFG.get("eval_results_file", "eval_results.json")
eval_df.to_json(eval_results_path, orient="records", indent=2, force_ascii=False)
print(f"✅ Results exported to {eval_results_path}")

# Also save as CSV for easy viewing
csv_path = results_dir / "eval_results.csv"  # ⬇️ CUSTOMIZE: change here if you want a different CSV filename
eval_df.to_csv(csv_path, index=False)
print(f"✅ CSV exported to {csv_path}")
